In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent
llm = ChatOpenAI(model='gpt-4o')

In [ ]:
loader = PyPDFLoader("../data/medical_report.pdf")
docs = loader.load()

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splitted_docs = splitter.split_documents(docs)

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = InMemoryVectorStore.from_documents(
    documents=splitted_docs, 
    embedding=embeddings
)

In [ ]:
same_record = vectorstore.similarity_search("patient name", k=1)
print(same_record[0].page_content)

### Agent = Tools, LLM, Prompt

In [ ]:
@tool
def reteriver_tool(query:str):
    """
        This tool can help you reterive the relavant data of the PDF document, and these pdf document have the details about medical reports.
    """
    print("Tool called: ", query)
    docs = vectorstore.similarity_search(query=query, k=4)
    context = ""

    for doc in docs:
        context = doc.page_content + "\n\n"

    return context

In [ ]:
reteriver_tool.invoke("Patient Name")

In [ ]:
System_Prompt="""
    You are a helpful assistant that answers questions using retrieved context.
	ALWAYS use the `retriever_tool` tool for questions requiring external knowledge.
"""

In [ ]:
agent = create_agent(
    model=llm,
    tools=[reteriver_tool],
    system_prompt=System_Prompt
)

In [ ]:
query = "What is the name of patient, and what is the name of Doctors"

In [ ]:
res = agent.invoke({"messages":[{"role":"user", "content":query}]})
res

In [ ]:
result = res["messages"][-1].content
print(result)